# Chapter 2 Exercise Solutions: Instructor Notebook

This Colab-style notebook accompanies the instructor guide for Chapter 2. It demonstrates the computational parts of the exercises: bootstrapped targets, empirical losses, target-network updates, replay and rollout buffers, stored log probabilities, learned dynamics losses, and simple algorithm-selection logic.

The notebook is intentionally small. It is not a full implementation of DQN, PPO, SAC, or model-based RL. Those algorithms are introduced in later chapters.


## 0. Environment setup

Run the next cell in Colab if the packages are not already installed. Local users can instead run `pip install -r requirements.txt`.


In [ ]:
# Colab setup. Uncomment if needed.
# !pip install -q numpy matplotlib torch gymnasium pytest


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

np.set_printoptions(precision=4, suppress=True)
print('Torch available:', TORCH_AVAILABLE)


## 1. Exercise 6 and 11: Q-learning targets and empirical squared loss

For a selected action, a one-step target has the form

$$y_i=r_i+\gamma d_i\max_{a'}Q_{ar	heta}(s_i^+,a'),$$

where $d_i=1$ for nonterminal transitions and $d_i=0$ for terminal transitions.


In [ ]:
def compute_q_targets(rewards, nonterminal, next_max_q, gamma):
    rewards = np.asarray(rewards, dtype=float)
    nonterminal = np.asarray(nonterminal, dtype=float)
    next_max_q = np.asarray(next_max_q, dtype=float)
    return rewards + gamma * nonterminal * next_max_q

def empirical_squared_loss(predicted_q, targets):
    predicted_q = np.asarray(predicted_q, dtype=float)
    targets = np.asarray(targets, dtype=float)
    return np.mean((predicted_q - targets) ** 2)

q_pred = np.array([1.2, 0.4, -0.1])
rewards = np.array([1.0, 0.0, -1.0])
nonterminal = np.array([1.0, 1.0, 0.0])
next_max_q = np.array([2.0, 1.0, 0.5])
gamma = 0.95

targets = compute_q_targets(rewards, nonterminal, next_max_q, gamma)
loss = empirical_squared_loss(q_pred, targets)
print('Targets:', targets)
print('Squared errors:', (q_pred - targets) ** 2)
print('Mean squared loss:', loss)


Expected result: targets $[2.90, 0.95, -1.00]$ and empirical squared loss approximately $1.3342$.


## 2. Exercise 7: Soft target-network update

The soft update $ar	heta\leftarrow	au	heta+(1-	au)ar	heta$ interpolates between a frozen target network and an immediate hard copy.


In [ ]:
theta = np.array([2.0, 4.0, 6.0])
theta_bar = np.array([0.0, 0.0, 0.0])

for tau in [0.0, 0.01, 0.1, 0.5, 1.0]:
    updated = tau * theta + (1.0 - tau) * theta_bar
    print(f'tau={tau:>4}: updated target = {updated}')


## 3. Exercise 8: Why store old log probabilities?

The old log probability is needed because the policy that generated the data may not equal the policy used later for optimization.


In [ ]:
def log_softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - logits.max()
    log_probs = shifted - np.log(np.exp(shifted).sum())
    return log_probs

old_logits = np.array([1.0, 0.5, -0.5])
new_logits = np.array([0.2, 1.3, -0.2])
action = 0

old_log_prob = log_softmax(old_logits)[action]
new_log_prob = log_softmax(new_logits)[action]
importance_ratio = np.exp(new_log_prob - old_log_prob)

print('Old log probability:', old_log_prob)
print('New log probability:', new_log_prob)
print('Importance ratio pi_new / pi_old:', importance_ratio)


If the old log probability is recomputed after the update, the denominator in the ratio is wrong. This is one reason rollout buffers store log probabilities at collection time.


## 4. Exercise 9: Supervised dynamics-model loss

A learned dynamics model can be trained as a supervised predictor of the next state. If reward is learned too, add a reward-prediction term.


In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(0)
    batch_size, state_dim, action_dim = 5, 3, 2
    model = nn.Sequential(nn.Linear(state_dim + action_dim, 32), nn.ReLU(), nn.Linear(32, state_dim + 1))

    s = torch.randn(batch_size, state_dim)
    a = torch.randn(batch_size, action_dim)
    true_next_s = torch.randn(batch_size, state_dim)
    true_r = torch.randn(batch_size, 1)

    pred = model(torch.cat([s, a], dim=-1))
    pred_next_s = pred[:, :state_dim]
    pred_r = pred[:, state_dim:]

    state_loss = F.mse_loss(pred_next_s, true_next_s)
    reward_loss = F.mse_loss(pred_r, true_r)
    total_loss = state_loss + reward_loss
    print('State prediction loss:', float(state_loss))
    print('Reward prediction loss:', float(reward_loss))
    print('Total model loss:', float(total_loss))
else:
    print('PyTorch not available in this runtime.')


## 5. Replay buffer and rollout buffer structures

This cell gives minimal versions of the two buffer types discussed in Chapter 2.


In [ ]:
from dataclasses import dataclass
import random

@dataclass
class Transition:
    state: object
    action: object
    reward: float
    next_state: object
    done: bool

class ReplayBuffer:
    def __init__(self, capacity, seed=0):
        self.capacity = capacity
        self.storage = []
        self.position = 0
        self.rng = random.Random(seed)

    def add(self, *transition):
        item = Transition(*transition)
        if len(self.storage) < self.capacity:
            self.storage.append(item)
        else:
            self.storage[self.position] = item
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return self.rng.sample(self.storage, batch_size)

class RolloutBuffer:
    def __init__(self):
        self.data = []

    def add(self, state, action, reward, done, log_prob, advantage):
        self.data.append({
            'state': state, 'action': action, 'reward': reward, 'done': done,
            'log_prob_at_collection': log_prob, 'advantage': advantage
        })

replay = ReplayBuffer(capacity=3)
for t in range(5):
    replay.add(t, t % 2, float(t), t + 1, t == 4)
print('Replay buffer length:', len(replay.storage))
print('Replay sample:', replay.sample(2))

rollout = RolloutBuffer()
rollout.add('s0', 'a0', 1.0, False, log_prob=-0.7, advantage=0.5)
print('Rollout item:', rollout.data[0])


## 6. Exercise 12--17: Rule-of-thumb algorithm selection helper

The helper below is not a replacement for engineering judgment. It shows how the task properties discussed in Chapter 2 can be converted into initial algorithm-family recommendations.


In [ ]:
def suggest_algorithm_families(action_space, simulator_available, online_interaction_safe,
                               offline_data_available, demonstrations_available,
                               safety_critical, long_horizon, multi_agent):
    suggestions = []
    if not online_interaction_safe:
        if demonstrations_available:
            suggestions.append('behavior cloning / imitation baseline')
        if offline_data_available:
            suggestions.append('conservative or behavior-constrained offline RL')
        if simulator_available:
            suggestions.append('simulation-based validation or model-based planning')
        return suggestions

    if action_space == 'discrete':
        suggestions.append('value-based DRL such as DQN variants')
    elif action_space == 'continuous':
        suggestions.append('continuous-control actor-critic such as PPO, TD3, or SAC')
    else:
        suggestions.append('actor-critic with a suitable action representation')

    if simulator_available:
        suggestions.append('model-based or planning-assisted RL')
    if safety_critical:
        suggestions.append('safe/constrained RL and shielded exploration')
    if long_horizon:
        suggestions.append('hierarchical or goal-conditioned RL')
    if multi_agent:
        suggestions.append('multi-agent RL')
    if demonstrations_available:
        suggestions.append('imitation pretraining followed by RL fine-tuning')
    return suggestions

mobile_robot = suggest_algorithm_families(
    action_space='continuous',
    simulator_available=True,
    online_interaction_safe=True,
    offline_data_available=False,
    demonstrations_available=False,
    safety_critical=True,
    long_horizon=True,
    multi_agent=False,
)
print('Mobile robot suggestions:')
for item in mobile_robot:
    print('-', item)

warehouse = suggest_algorithm_families(
    action_space='discrete',
    simulator_available=False,
    online_interaction_safe=False,
    offline_data_available=True,
    demonstrations_available=True,
    safety_critical=True,
    long_horizon=False,
    multi_agent=False,
)
print('
Warehouse suggestions:')
for item in warehouse:
    print('-', item)


## 7. Instructor checklist

When grading Chapter 2 exercises, prefer answers that connect the taxonomy to the learning signal and data regime. Strong answers distinguish:

- true environment interaction from imagined/model-generated rollouts;
- on-policy rollout distributions from replay-buffer distributions;
- fixed offline datasets from growing online replay buffers;
- action selection by an actor from evaluation by a critic;
- average return from safety, robustness, and tail-risk metrics.
